In [1]:
!git clone https://github.com/Vedant988/tmz-R2.git

Cloning into 'tmz-R2'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 76 (delta 30), reused 60 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 36.98 KiB | 3.08 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [1]:
%cd /content/tmz-R2
!pip install -r requirements.txt

/content/tmz-R2
  Cloning https://github.com/VarunGumma/IndicTransToolkit.git to /tmp/pip-install-15n1j5t8/indictranstoolkit_c40776be08924f038934a9e675334c73
  Running command git clone --filter=blob:none --quiet https://github.com/VarunGumma/IndicTransToolkit.git /tmp/pip-install-15n1j5t8/indictranstoolkit_c40776be08924f038934a9e675334c73
  Resolved https://github.com/VarunGumma/IndicTransToolkit.git to commit 3efb8418d0721b4ce267c2b3586899d313191357
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
%cd /content/tmz-R2
!git pull

/content/tmz-R2
Already up to date.


# cell 1

In [3]:
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm
import sacrebleu
from google.colab import userdata
import os

# Authenticate using your Colab Secret
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN secret not found. Make sure you added it to the Secrets sidebar.")

# Device configuration (Uses T4 GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load IndicTrans2 (1B Parameter Model)
model_name = "ai4bharat/indictrans2-en-indic-1B"

# trust_remote_code=True is required for AI4Bharat's custom tokenization
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# We use torch.float16 for faster inference and lower VRAM usage on T4
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(device)

print("IndicTrans2-1B loaded successfully!")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/4.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

IndicTrans2-1B loaded successfully!


# Cell 2

In [5]:
from IndicTransToolkit.processor import IndicProcessor

# Initialize the processor
ip = IndicProcessor(inference=True)

def translate_en_to_ta(text):
    src_lang = "eng_Latn"
    tgt_lang = "tam_Taml"

    # 1. Preprocess the text (this normalizes the text and safely prepends the language tags)
    processed_text = ip.preprocess_batch([text], src_lang=src_lang, tgt_lang=tgt_lang)

    # 2. Tokenize
    inputs = tokenizer(
        processed_text,
        truncation=True,
        padding="longest",
        return_tensors="pt",
        return_attention_mask=True
    ).to(device)

    # 3. Generate translation (No need for forced_bos_token_id anymore!)
    with torch.inference_mode():
        translated_tokens = model.generate(
            **inputs,
            max_length=256,
            num_beams=5
        )

    # 4. Decode
    decoded_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

    # 5. Postprocess (cleans up any remaining artifacts for a clean BLEU score)
    return ip.postprocess_batch(decoded_text, lang=tgt_lang)[0]

# Test translation
sample = "Hello, how are you? Welcome to the Indic Translation project."
print("Sample Translation:", translate_en_to_ta(sample))

Sample Translation: வணக்கம், எப்படி இருக்கிறீர்கள்? இந்திய மொழிபெயர்ப்பு திட்டத்திற்கு வரவேற்கிறோம்.


# Cell 3

In [6]:
data = {
    'english_source': ["This is a test sentence.", "Machine learning is fascinating.", "I love programming."],
    'tamil_reference': ["இது ஒரு சோதனை வாக்கியம்.", "இயந்திர கற்றல் கவர்ச்சிகரமானது.", "எனக்கு நிரலாக்கம் பிடிக்கும்."]
}
df = pd.DataFrame(data)

df = pd.read_csv('data/raw/translation_dataset.csv')

tqdm.pandas(desc="Translating")
df['indictrans2_prediction'] = df['english_source'].progress_apply(translate_en_to_ta)

# Evaluate using SacreBLEU
bleu_score = sacrebleu.corpus_bleu(
    df['indictrans2_prediction'].tolist(),
    [df['tamil_reference'].tolist()]
)

print(f"SacreBLEU Score: {bleu_score.score:.2f}")

display(df[['english_source', 'indictrans2_prediction', 'tamil_reference']])

# Save the outputs according to your structure
df.to_csv('task1_translation_evaluation/part_a_batch_translation/translation_outputs.csv', index=False)


Translating: 100%|██████████| 15/15 [00:04<00:00,  3.60it/s]

SacreBLEU Score: 42.49


,english_source,indictrans2_prediction,tamil_reference
0,This is a test sentence.,இது ஒரு சோதனை வாக்கியம்.,இது ஒரு சோதனை வாக்கியம்.
1,Machine learning is fascinating.,இயந்திர கற்றல் சுவாரஸ்யமானது.,இயந்திர கற்றல் கவர்ச்சிகரமானது.
2,I love programming.,எனக்கு புரோகிராமிங் பிடிக்கும்.,எனக்கு நிரலாக்கம் பிடிக்கும்.
3,Artificial intelligence is transforming the wo...,செயற்கை நுண்ணறிவு உலகை மாற்றியமைத்து வருகிறது.,செயற்கை நுண்ணறிவு உலகத்தை மாற்றிக் கொண்டிருக்க...
4,The weather is beautiful today.,இன்று வானிலை அழகாக உள்ளது.,இன்று வானிலை அழகாக இருக்கிறது.
5,Please let me know if you need any help.,உங்களுக்கு ஏதேனும் உதவி தேவைப்பட்டால் எனக்கு த...,உங்களுக்கு ஏதாவது உதவி தேவைப்பட்டால் தயவுசெய்த...
6,He is reading a book in the library.,நூலகத்தில் ஒரு புத்தகத்தைப் படிக்கிறார்.,அவர் நூலகத்தில் ஒரு புத்தகம் படித்துக் கொண்டிர...
7,We are going to the market to buy vegetables.,காய்கறிகள் வாங்க சந்தைக்குச் செல்கிறோம்.,நாங்கள் காய்கறிகள் வாங்க சந்தைக்குச் செல்கிறோம்.
8,The quick brown fox jumps over the lazy dog.,வேகமாக பழுப்பு நிற நரி சோம்பேறி நாய் மீது குதி...,வேகமான பழுப்பு நரி சோம்பேறி நாயின் மீது குதிக்...
9,Data structures and algorithms are essential f...,மென்பொருள் பொறியாளர்களுக்கு தரவு கட்டமைப்புகள்...,மென்பொருள் பொறியாளர்களுக்கு தரவு கட்டமைப்புகள்...


In [7]:
# Save SacreBLEU score to match the required repo structure
bleu_df = pd.DataFrame({'metric': ['SacreBLEU'], 'score': [bleu_score.score]})
bleu_df.to_csv('task1_translation_evaluation/part_a_batch_translation/sacrebleu_results.csv', index=False)
print(" sacrebleu_results.csv saved!")

 sacrebleu_results.csv saved!


---

# part_b_token_eda.ipynb

In [8]:
import torch
import gc
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
tqdm.pandas()

print("--- 1. Generating Translations with NLLB-200 ---")
nllb_tokenizer = AutoTokenizer.from_pretrained('facebook/nllb-200-distilled-600M')
nllb_model = AutoModelForSeq2SeqLM.from_pretrained('facebook/nllb-200-distilled-600M').to(device)

def translate_nllb(text):
    inputs = nllb_tokenizer(text, return_tensors="pt").to(device)
    # NLLB requires forcing the target language token
    forced_bos_token_id = nllb_tokenizer.lang_code_to_id["tam_Taml"]
    with torch.inference_mode():
        outputs = nllb_model.generate(**inputs, forced_bos_token_id=forced_bos_token_id, max_length=128)
    return nllb_tokenizer.decode(outputs[0], skip_special_tokens=True)

df['nllb_prediction'] = df['english_source'].progress_apply(translate_nllb)

# Clear VRAM so we don't crash when loading mT5
del nllb_model
del nllb_tokenizer
torch.cuda.empty_cache()
gc.collect()

print("\n--- 2. Generating Translations with mT5-Base ---")
mt5_tokenizer = AutoTokenizer.from_pretrained('google/mt5-base')
mt5_model = AutoModelForSeq2SeqLM.from_pretrained('google/mt5-base').to(device)

def translate_mt5(text):
    # mT5 works best with a prompt prefix
    inputs = mt5_tokenizer(f"translate English to Tamil: {text}", return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = mt5_model.generate(**inputs, max_length=128)
    return mt5_tokenizer.decode(outputs[0], skip_special_tokens=True)

df['mt5_prediction'] = df['english_source'].progress_apply(translate_mt5)

# Clear VRAM again
del mt5_model
del mt5_tokenizer
torch.cuda.empty_cache()
gc.collect()

# Save the predictions back to the CSV so cell [6] can read them
df.to_csv('task1_translation_evaluation/part_a_batch_translation/translation_outputs.csv', index=False)
print("\n All translations generated and saved successfully!")

--- 1. Generating Translations with NLLB-200 ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
100%|██████████| 15/15 [00:05<00:00,  2.56it/s]



--- 2. Generating Translations with mT5-Base ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

...

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:515: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

100%|██████████| 15/15 [00:02<00:00,  6.04it/s]



 All translations generated and saved successfully!


In [9]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

# 1. Ensure directories exist
os.makedirs('task1_translation_evaluation/part_b_token_analysis/plots', exist_ok=True)

# 2. Load the dataset (now containing all 3 predictions)
df = pd.read_csv('task1_translation_evaluation/part_a_batch_translation/translation_outputs.csv')

# 3. Load Tokenizers
print("Loading tokenizers for comparison...")
tokenizers = {
    'IndicTrans2': AutoTokenizer.from_pretrained('ai4bharat/indictrans2-en-indic-1B', trust_remote_code=True),
    'NLLB-200': AutoTokenizer.from_pretrained('facebook/nllb-200-distilled-600M'),
    'mT5-Base': AutoTokenizer.from_pretrained('google/mt5-base')
}

def calculate_metrics(text, tokenizer, model_name, is_target=False):
    text_str = str(text)
    words = text_str.split()
    num_words = len(words)

    # Metric 1: Average Word Length
    avg_word_length = sum(len(w) for w in words) / num_words if num_words > 0 else 0

    # Tokenization handling
    if model_name == 'IndicTrans2':
        formatted = f"tam_Taml eng_Latn {text_str}" if is_target else f"eng_Latn tam_Taml {text_str}"
        tokens = tokenizer.encode(formatted)[2:] # Strip language tags
    else:
        tokens = tokenizer.encode(text_str)

    num_tokens = len(tokens)
    unk_token_id = tokenizer.unk_token_id

    # Metric 2: Subword Fragmentation (Tokens per Word)
    subword_fragmentation = num_tokens / num_words if num_words > 0 else 0

    # Metric 3: Unknown Token Rate
    unk_count = tokens.count(unk_token_id) if unk_token_id is not None else 0
    unknown_token_rate = unk_count / num_tokens if num_tokens > 0 else 0

    return num_tokens, avg_word_length, subword_fragmentation, unknown_token_rate

# 4. Process the metrics across all models
results = []
for idx, row in df.iterrows():
    eng_text = row['english_source']

    for model_name, tokenizer in tokenizers.items():
        # Dynamically select the correct prediction
        if model_name == 'IndicTrans2':
            tam_text = row['indictrans2_prediction']
        elif model_name == 'NLLB-200':
            tam_text = row['nllb_prediction']
        elif model_name == 'mT5-Base':
            tam_text = row['mt5_prediction']

        # Calculate metrics using the correct tam_text
        src_tokens, src_avg_len, src_frag, src_unk = calculate_metrics(eng_text, tokenizer, model_name, is_target=False)
        tgt_tokens, tgt_avg_len, tgt_frag, tgt_unk = calculate_metrics(tam_text, tokenizer, model_name, is_target=True)

        expansion_ratio = tgt_tokens / src_tokens if src_tokens > 0 else 0

        results.append({
            'sentence_id': idx,
            'model': model_name,
            'source_token_count': src_tokens,
            'target_token_count': tgt_tokens,
            'expansion_ratio': expansion_ratio,
            'avg_word_length': tgt_avg_len,
            'subword_fragmentation': tgt_frag,
            'unknown_token_rate': tgt_unk
        })

# Save the master dataset
results_df = pd.DataFrame(results)
results_df.to_csv('task1_translation_evaluation/part_b_token_analysis/multi_model_metrics.csv', index=False)
print("✅ multi_model_metrics.csv saved with all required columns!")

# --- PLOTS GENERATION ---

# Plot 1: Scatter Plot (Source vs Target Tokens)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=results_df, x='source_token_count', y='target_token_count', hue='model', style='model', s=100)
plt.title('Scatter Plot: Source vs. Target Token Counts')
plt.xlabel('English Source Token Count')
plt.ylabel('Tamil Target Token Count')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig('task1_translation_evaluation/part_b_token_analysis/plots/scatter_source_vs_target.png', dpi=300, bbox_inches='tight')
plt.close()

# Plot 2: Bar Chart (Average Expansion per Model)
plt.figure(figsize=(8, 6))
avg_exp = results_df.groupby('model')['expansion_ratio'].mean().reset_index()
sns.barplot(data=avg_exp, x='model', y='expansion_ratio', hue='model', palette='magma', legend=False)
plt.title('Average Token Expansion Ratio by Model (English -> Tamil)')
plt.xlabel('Model')
plt.ylabel('Expansion Ratio (Target / Source)')
for i, val in enumerate(avg_exp['expansion_ratio']):
    plt.text(i, val + 0.05, f'{val:.2f}', ha='center', fontweight='bold')
plt.savefig('task1_translation_evaluation/part_b_token_analysis/plots/bar_avg_expansion.png', dpi=300, bbox_inches='tight')
plt.close()

print("✅ Scatter plot and Bar chart successfully generated and saved!")

Loading tokenizers for comparison...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:515: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


✅ multi_model_metrics.csv saved with all required columns!
✅ Scatter plot and Bar chart successfully generated and saved!


In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure results_df is in memory from the previous cell
if 'results_df' not in locals():
    results_df = pd.read_csv('task1_translation_evaluation/part_b_token_analysis/multi_model_metrics.csv')

plot_dir = 'task1_translation_evaluation/part_b_token_analysis/plots'

# --- 1. Generate Missing EDA Plots ---

# Plot 3: Histogram of Target Token Counts
plt.figure(figsize=(10, 6))
sns.histplot(data=results_df, x='target_token_count', hue='model', element='step', kde=True, bins=15, alpha=0.6)
plt.title('Histogram: Target Token Counts across Models')
plt.xlabel('Tamil Target Token Count')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig(f'{plot_dir}/histogram_token_counts.png', dpi=300, bbox_inches='tight')
plt.close()

# Plot 4: Boxplot of Expansion Ratios
plt.figure(figsize=(10, 6))
sns.boxplot(data=results_df, x='model', y='expansion_ratio', hue='model', palette='Set2', legend=False)
plt.title('Boxplot: Token Expansion Ratios by Model')
plt.xlabel('Model')
plt.ylabel('Expansion Ratio (Target / Source)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig(f'{plot_dir}/boxplot_expansion_ratios.png', dpi=300, bbox_inches='tight')
plt.close()

print("✅ Missing Histogram and Boxplot successfully generated and saved!")

# --- 2. Save Missing CSV Files for Repo Structure ---

# Extract and save token_counts.csv
token_counts_df = results_df[['sentence_id', 'model', 'source_token_count', 'target_token_count']]
token_counts_df.to_csv('task1_translation_evaluation/part_b_token_analysis/token_counts.csv', index=False)

# Extract and save engineered_features.csv
features_df = results_df[['sentence_id', 'model', 'expansion_ratio', 'avg_word_length', 'subword_fragmentation', 'unknown_token_rate']]
features_df.to_csv('task1_translation_evaluation/part_b_token_analysis/engineered_features.csv', index=False)

print("✅ token_counts.csv and engineered_features.csv successfully separated and saved!")

# --- 3. Identify Lowest Expansion Rate for Analysis ---

avg_exp = results_df.groupby('model')['expansion_ratio'].mean()
lowest_model = avg_exp.idxmin()
lowest_rate = avg_exp.min()

print(f"\n📊 ANALYSIS RESULT:")
print(f"The model with the lowest average expansion rate is: **{lowest_model}** (Rate: {lowest_rate:.2f})")

✅ Missing Histogram and Boxplot successfully generated and saved!
✅ token_counts.csv and engineered_features.csv successfully separated and saved!

📊 ANALYSIS RESULT:
The model with the lowest average expansion rate is: **mT5-Base** (Rate: 0.52)


In [ ]:
import shutil
from google.colab import files

# 1. Define the directory to zip and the output path
dir_to_zip = '/content/tmz-R2/task1_translation_evaluation/part_b_token_analysis'
output_zip = '/part_b_token_analysis_backup'

# 2. Create the zip archive
shutil.make_archive(output_zip, 'zip', dir_to_zip)

# 3. Download the file to your local computer
files.download(f"{output_zip}.zip")
print("✅ Part B analysis folder successfully zipped and download started!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Part B analysis folder successfully zipped and download started!


---

# part_c

In [11]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

# 1. Create the Part C directory structure
base_dir = 'task1_translation_evaluation/part_c_indic_token_behavior'
os.makedirs(f'{base_dir}/plots', exist_ok=True)

# 2. Load tokenizers for comparison
print("Loading Tokenizers...")
tokenizers = {
    'IndicTrans2': AutoTokenizer.from_pretrained('ai4bharat/indictrans2-en-indic-1B', trust_remote_code=True),
    'NLLB-200': AutoTokenizer.from_pretrained('facebook/nllb-200-distilled-600M'),
    'mT5-Base': AutoTokenizer.from_pretrained('google/mt5-base')
}

# Load the predictions
df = pd.read_csv('task1_translation_evaluation/part_a_batch_translation/translation_outputs.csv')

# We use IndicTrans2's high-quality Tamil predictions as our "gold standard" to test all tokenizers on
tamil_sentences = df['indictrans2_prediction'].dropna().tolist()

behavior_data = []
pattern_data = []

print("Analyzing token behavior across models...")
for idx, text in enumerate(tamil_sentences):
    text_str = str(text)
    chars_no_spaces = len(text_str.replace(" ", ""))
    words = text_str.split()
    num_words = max(1, len(words))

    for model_name, tokenizer in tokenizers.items():
        # Handle specific formatting requirements
        if model_name == 'IndicTrans2':
            formatted = f"tam_Taml eng_Latn {text_str}"
            token_ids = tokenizer.encode(formatted)[2:] # Strip language tags
        else:
            token_ids = tokenizer.encode(text_str)

        num_tokens = max(1, len(token_ids))
        token_pieces = [tokenizer.decode([tid]) for tid in token_ids]

        # --- Calculate Required Metrics ---

        # 1. Characters per token & Subword Splitting
        chars_per_token = chars_no_spaces / num_tokens
        tokens_per_word = num_tokens / num_words

        # 2. Memory Footprint: Self-attention scales quadratically O(N^2)
        memory_footprint = num_tokens ** 2

        # 3. Unicode Fragmentation & Rare Tokens
        # (Looking for raw byte fallbacks like <0xE0> or Unknown tokens)
        unk_id = tokenizer.unk_token_id
        unk_count = token_ids.count(unk_id) if unk_id is not None else 0
        byte_fallback_count = sum(1 for piece in token_pieces if piece.startswith('<0x'))
        fragmentation_score = unk_count + byte_fallback_count

        behavior_data.append({
            'sentence_id': idx,
            'model': model_name,
            'num_tokens': num_tokens,
            'chars_per_token': chars_per_token,
            'tokens_per_word': tokens_per_word,
            'memory_footprint_O_N2': memory_footprint,
            'unicode_frag_count': fragmentation_score
        })

        # Save pattern breakdown for the first 3 sentences to illustrate the splits
        if idx < 3:
            pattern_data.append({
                'sentence_id': idx,
                'model': model_name,
                'tamil_sentence': text_str,
                'token_breakdown': " | ".join(token_pieces)
            })

# Save datasets
behavior_df = pd.DataFrame(behavior_data)
pattern_df = pd.DataFrame(pattern_data)

behavior_df.to_csv(f'{base_dir}/indic_token_metrics.csv', index=False)
pattern_df.to_csv(f'{base_dir}/tamil_token_patterns.csv', index=False)
print("✅ Metrics and Token Patterns saved!")

# --- PLOTS ---

# Plot 1: Characters per Token (Boxplot)
plt.figure(figsize=(10, 6))
sns.boxplot(data=behavior_df, x='model', y='chars_per_token', hue='model', palette='Set2')
plt.title('Average Characters per Token (Semantic Density)')
plt.ylabel('Characters / Token')
plt.savefig(f'{base_dir}/plots/chars_per_token_boxplot.png', dpi=300)
plt.close()

# Plot 2: Memory Footprint Estimation (Barplot)
plt.figure(figsize=(10, 6))
sns.barplot(data=behavior_df, x='model', y='memory_footprint_O_N2', hue='model', estimator='mean', palette='Reds', errorbar=None)
plt.title('Average Transformer Memory Footprint $O(N^2)$')
plt.ylabel('Attention Matrix Size (Tokens²)')
plt.savefig(f'{base_dir}/plots/memory_footprint.png', dpi=300)
plt.close()

print("✅ Plots successfully generated and saved!")

# Display the split pattern for the first sentence
print("\n--- Example: How different models split the same Tamil sentence ---")
display(pattern_df[pattern_df['sentence_id'] == 0][['model', 'token_breakdown']])

Loading Tokenizers...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:515: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Analyzing token behavior across models...
✅ Metrics and Token Patterns saved!
✅ Plots successfully generated and saved!

--- Example: How different models split the same Tamil sentence ---


,model,token_breakdown
0,IndicTrans2,थेके | प्रोडक्ट्स | निकास | प्रशंसक | थेके | स...
1,NLLB-200,eng_Latn | இது | ஒரு | சோ | தனை | வா | க்கிய |...
2,mT5-Base,இது | ஒரு | சோ | தனை | வாக்க | ியம் | . | </s>
